### Linear MPC



#### Задача 1

Выведите линейный MPC для следования траектории $X = \{x_{i+1}^*\}_{i=0}^{N-1}$. Вывод слегка обобщает лекционный результат. 

YOUR MARKDOWN HERE

#### Задача 2

Реализуйте функцию generate_controls, следуя выводу из первой задачи. Она должна генерировать для заданной системы последовательность управлений, оптимальную в смысле квадратичной функции стоимости.

In [ ]:
import numpy as np

def generate_controls(x_0, X_star, A, B, N, Q, R):
    #A_hat = np.zeros(...)
    
    #YOUR CODE BELOW
    
    #YOUR CODE ABOVE
        
    return U

A = np.array\
        ([[0, 1, 0,   0],
          [0, 0, 0.5, 0],
          [0, 0, 0,   1],
          [0, 0, 0.4, 0]])

B = np.array([[0], [0.5], [0], [1.0]])

Q = np.eye(4)

R = np.eye(1)

horizon = 3

x_0 = np.array([[10], [1], [2], [1]])

X_star = ...

print(generate_controls(x_0, X_star, A, B, horizon, Q, R))

#### Задача 3

Реализуйте функцию generate_controls, поставив задачу поиска оптимальной последовательности управлений как задачу квадратичного программирования и решив ее с помощью *solve_qp*. Учтите ограничения на управление (F_max).

In [ ]:
import numpy as np
from qpsolvers import solve_qp

def generate_controls(x_0, X_star, A, B, N, Q, R, u_max):
    #YOUR CODE BELOW
    
    #YOUR CODE ABOVE
    
    U = solve_qp(C, d, E, f, solver="proxqp")
    
    return U

A = np.array\
        ([[0, 1, 0,   0],
          [0, 0, 0.5, 0],
          [0, 0, 0,   1],
          [0, 0, 0.4, 0]])

B = np.array([[0], [0.5], [0], [1.0]])

Q = np.eye(4)

R = np.eye(1)

horizon = 3

F_max = 2

x_0 = np.array([[10], [1], [2], [1]])
X_star = ...

print(generate_controls(x_0, X_star, A, B, horizon, Q, R, F_max))

#### Задача 4

Примените функцию generate_controls для управления динамической точкой с массой. Визуализируйте результат.

In [ ]:
# YOUR CODE HERE

#### Задача 5

Примените функцию generate_controls для управления линеаризованным Cart-pole. Ниже дана заготовка кода, ее нужно будет дополнить.

Система должна совершать колебания вокруг координаты, которую вы задаете на трекбаре.

Управление должно осуществляться с помощью MPC для линейной системы.

ВАЖНО! Управление должно быть согласно линейной модели, а вот интегрирование должно быть для полной (нелинейной) модели.

Рекомендуемый порядок действий:
- получите лагранжиан
- получите уравнения движения, добавьте в них управление
- выразите из них x_ddot, theta_ddot
- линейризуйте систему в окрестности верхнего положения равновесия
- запрогайте это, дописав матрицы в классе
- встройте в цикл симуляции генерацию управления согласно MPC
- сдвигайте ноль согласно положению, задаваемому трекбаром

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from scipy import linalg
import copy
import math

def draw_cart_pole(cx, cy, x, alpha, scale = 100, color = (234, 123, 123), canvas = None,
                   alpha_0 = 0):
    if (canvas is None):
        canvas = np.ones((700, 700, 3)) * 0
    
    h, w, _ = canvas.shape
    
    cv2.line(canvas, (0, h // 2), (w, h // 2), color, 2)
    
    cv2.circle(canvas, (w // 2, h // 2), 7, color, 2)
    
    cv2.rectangle(canvas, (int(cx - w // 20 + x * scale), cy - h // 40),
                          (int(cx + w // 20 + x * scale), cy + h // 40), color, -1)
    
    cv2.line(canvas, (int(cx + x * scale), cy),
                     (int(cx + x * scale - w / 5 * math.sin(alpha + alpha_0)),
                          cy - int(w / 5 * math.cos(alpha + alpha_0))), color, 2)
    
    cv2.circle(canvas, (int(cx + x * scale - w / 5 * math.sin(alpha + alpha_0)),
                          cy - int(w / 5 * math.cos(alpha + alpha_0))), 27, color, 7)
    
    return canvas

class Cart_pole:
    def __init__(self, M = 1.0, m = 1.0, l = 3.0, g = 10.0,
                 a0 = 0.2, ad0 = -0.3, x0 = 1.2, xd0 = -0.1,
                 dt = 0.001, Q = np.eye(4), R = np.eye(1),
                 WIND_X = 700, scale = 100, max_goal_pos = 3):
        self.M = M
        self.m = m
        self.l = l
        self.g = g
        
        self.x = np.array([[x0], [xd0], [a0], [ad0]])
        self.dt = dt
                
        #YOUR CODE BELOW
        self.A = np.array()
        
        self.B = np.array()
        #YOUR CODE BELOW
        
        self.Q = Q
        self.R = R
                
        self.WIND_X = WIND_X
        self.scale = scale
        self.target_x = 0.0
        self.max_goal_pos = max_goal_pos
        
        cv2.namedWindow('cart_pole')
        cv2.createTrackbar("goal_position", "cart_pole", max_goal_pos,
                           max_goal_pos * 2,
                          lambda i : i)

    def get_state(self):
        return self.x
    
    def propagate_system(self, u):
        x  = self.x[0, 0]
        xd = self.x[1, 0]
        a  = self.x[2, 0]
        ad = self.x[3, 0]
        
        g = self.g
        M = self.M
        m = self.m
        l = self.l
        
        add = (g * (M + m) * math.sin(a) + (u - m * l * ad**2 * math.sin(a)) * math.cos(a)) / (l * (M + m * math.sin(a)**2))
        xdd = 1.0 / (M + m) * (u + m * l * add * math.cos(a) - m * l * ad**2 * math.sin(a))
        
        self.x[1, 0] += xdd * self.dt
        self.x[0, 0] += self.x[1, 0] * self.dt
        
        self.x[3, 0] += add * self.dt
        self.x[2, 0] += self.x[3, 0] * self.dt

def state_action_cost(x, u, Q, R):
    cost = x.T @ Q @ x + u.T @ R @ u
    
    return cost

def episode_cost(x_hist, u_hist, Q, R):
    total_cost = 0
    cost_hist = []
    
    for x, u in zip(x_hist, u_hist):
        cost = state_action_cost(x, u, Q, R)
        
        total_cost += cost
        cost_hist.append(cost)
    
    return total_cost, cost_hist

def run_cart_pole_episode(Q, R, a0 = 0.2, ad0 = -0.3, x0 = 1.2, xd0 = -0.1,
                          scale = 30, max_goal_pos = 10):
    WIND_X = 700
    WIND_Y = 700
    canvas = np.ones((700, 700, 3), np.uint8) * 70
    
    dyn_point = Cart_pole(M = 1.0, m = 1.0, l = 1.0, g = 10.0,
                 a0 = a0, ad0 = ad0, x0 = x0, xd0 = xd0,
                 dt = 0.15, Q = Q, R = R, WIND_X = WIND_X,
                 scale = scale, max_goal_pos = max_goal_pos)

    iter_num = 750000
    i = 0

    x_traj = []
    v_traj = []
    u_traj = []
    
    Q = np.eye(4) * 1
        
    R = np.eye(1)
    
    horizon = 10

    F_max = 20
    
    def get_A(dt):
        M = 1.0
        m = 1.0
        l = 1.0
        g = 10.0
        
        return np.array ...
    
    while(True):
        state = dyn_point.get_state()
        
        target = cv2.getTrackbarPos("goal_position", "cart_pole")
        
        
        #controls = generate_controls(state - np.array([[target - max_goal_pos + 7 * math.sin(i / 5.0)],
        #                            [0], [0], [0]]), get_A(0.5), dyn_point.B * 0.5, horizon, Q, R, F_max)
        
        # YOUR CODE BELOW
        
        controls = 
        
        # YOUR CODE ABOVE
        
        control = controls[0]
        
        dyn_point.propagate_system(control)
                
        x_traj.append(state)
        u_traj.append(control)
        
        canvas = cv2.addWeighted(canvas, 0.85, canvas, 0, 0)
        draw_cart_pole(WIND_X // 2, WIND_Y // 2, state[0, 0], state[2, 0],
                       canvas = canvas, scale = scale)

        cv2.imshow("cart_pole", canvas)
        
        i += 1

        if (i > iter_num):
            break
        
        key = cv2.waitKey(10) & 0xFF
        
        if (key == ord('q')):
            break
    
    cv2.destroyAllWindows()
    cv2.waitKey(10)
    
    return x_traj, u_traj

Q = np.eye(4)
Q[0, 0] *= 10
R = np.eye(1) * 1

x_hist, u_hist = run_cart_pole_episode(Q, R, a0 = -0.2, ad0 = 0.01,
                x0 = 0.4, xd0 = 0.1, max_goal_pos = 10)